<a href="https://colab.research.google.com/github/amandaliram/pln_nlp_repository/blob/main/A12_RP09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Roteiro Prático 09 - Aprendizagem Aplicada II**

### **1. Objetivos de Aprendizagem**
Ao final deste guia, você será capaz de:
* **Implementar** um classificador probabilístico Multinomial Naive Bayes para categorização de sentimentos em pipelines de e-commerce.
* **Validar** o desempenho do modelo através de matrizes de confusão e métricas de desempenho (Acurácia, Precisão, Recall e F1-Score) aprendidas na Aula 13.
* **Analisar** criticamente o impacto da suposição de independência dos termos na velocidade de processamento e na precisão semântica.
* **Otimizar** a representação de dados utilizando a técnica TF-IDF (Aula 06) para atenuar o ruído de termos ubíquos e destacar palavras-chave informativas.

---

### **2. Fundamentação Teórica**

**2.1 A Anatomia do Erro (Matriz de Confusão)**  

A base de toda a avaliação reside em quatro estados fundamentais:
* **Verdadeiro Positivo (VP):** O modelo previu "Crítica" e o texto era realmente uma "Crítica".
* **Verdadeiro Negativo (VN):** O modelo previu "Elogio" e o texto era realmente um "Elogio".
* **Falso Positivo (FP):** O erro de "Alarme Falso". O modelo diz que é "Crítica", mas era um "Elogio".
* **Falso Negativo (FN):** O erro de "Omissão". O modelo diz que é "Elogio", mas o cliente estava furioso.

**2.2 A Matemática das métricas**  

* **Acurácia (Accuracy):** O percentual de acertos totais sobre o conjunto de dados.
  $$\text{Acurácia} = \frac{VP + VN}{VP + VN + FP + FN}$$
* **Precisão (Precision):** A proporção de instâncias que o sistema classificou como positivas e que são verdadeiramente positivas. Mede a "confiabilidade" quando o modelo alerta algo.
  $$\text{Precisão} = \frac{VP}{VP + FP}$$
* **Recall (Revocação):** A proporção de itens positivos reais que o sistema efetivamente conseguiu detectar. Mede a capacidade do modelo de não deixar escapar casos positivos.
  $$\text{Recall} = \frac{VP}{VP + FN}$$
* **F1-Score (Medida-F):** A média harmônica entre a Precisão e o Recall, utilizada para otimização unificada.
  $$\text{F1-Score} = 2 \times \frac{\text{Precisão} \times \text{Recall}}{\text{Precisão} + \text{Recall}}$$

**2.3 Lista de Vocabulário**  

* **Ground Truth:** O rótulo real/correto de um dado, usado como referência para a avaliação.
* **Trade-off:** O equilíbrio necessário onde, ao tentar aumentar a Precisão (ser mais conservador), acabamos por diminuir o Recall (deixar passar mais casos).
* **Classification Report:** Função do Scikit-Learn que gera um sumário textual com as métricas de Precisão, Recall e F1 para cada classe.
* **Data Leakage (Vazamento):** Erro grave onde informações do conjunto de treino "escapam" para o teste, gerando métricas falsamente perfeitas.
* **Macro Average:** Média das métricas calculada de forma independente para cada classe, tratando-as com o mesmo peso (ideal para identificar falhas em classes minoritárias).

### **3. Exemplo de classificação de textos**

**3.1 O Problema: Classificação de Spam**  

Imagine que o nosso objetivo é criar um sistema que consiga classificar e-mails automaticamente como spam ou não-spam. Para isso, vamos usar um conjunto de dados simples:
* `['Compre agora e ganhe um desconto!', 'spam']`
* `['Olá, tudo bem? Vamos nos encontrar?', 'não-spam']`

O nosso trabalho é pegar esses dados e, com os algoritmos, criar um modelo que aprenda a diferença entre os dois tipos de e-mail.

**3.2 Preparação dos Dados e Vetorização**  

Antes de treinar o modelo, precisamos preparar os dados. Vamos usar a biblioteca scikit-learn para vetorizar o texto, ou seja, converter as palavras em números.  

**Tarefa 3.2:** Crie um conjunto de dados e vetorize-o com TF-IDF.

In [29]:
# 1. Instalação e importação das bibliotecas necessárias
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [30]:
# 2. Preparação dos Dados e Vetorização
# Conjunto de dados de exemplo
dados = {
    'texto': [
        'Compre agora e ganhe um desconto!',
        'Olá, tudo bem? Vamos nos encontrar?',
        'Ganhe dinheiro fácil e rápido, sem esforço.',
        'Convite para um café, me responda por favor.',
        'Promoção imperdível! Clique para ganhar um brinde.',
        'Reunião agendada para amanhã, 10h.'
    ],
    'categoria': [
        'spam',
        'não-spam',
        'spam',
        'não-spam',
        'spam',
        'não-spam'
    ]
}

In [31]:
# Criando um DataFrame para organizar os dados
df = pd.DataFrame(dados)

# Vetorização do texto com TF-IDF
# O TfidfVectorizer transforma o texto em vetores de relevância
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['texto'])

# O 'y' são as categorias que o nosso modelo vai aprender
y = df['categoria']

print("Dados vetoriais (primeiro documento):")
print(X.toarray()[0])

Dados vetoriais (primeiro documento):
[0.        0.        0.490779  0.        0.        0.        0.
 0.        0.490779  0.        0.490779  0.        0.        0.
 0.        0.        0.        0.4024458 0.        0.        0.
 0.        0.        0.        0.        0.        0.        0.
 0.        0.        0.3397724 0.       ]


**3.3 Treinamento do Modelo: Naive Bayes e SVM**  

Agora que o nosso texto está em formato numérico, vamos treinar os modelos de classificação.
  

**Tarefa 3.3:** Treine os modelos Naive Bayes e SVM.

In [32]:
# 3. Treinamento do Modelo: Naive Bayes e SVM

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

# Instanciando e treinando o modelo Naive Bayes
modelo_nb = MultinomialNB()
modelo_nb.fit(X_treino, y_treino)

# Instanciando e treinando o modelo SVM
modelo_svm = SVC(kernel='linear')
modelo_svm.fit(X_treino, y_treino)

# Pergunta para reflexão: O que o método '.fit()' faz?
# Resposta: O '.fit()' é o momento em que o modelo 'aprende' a partir dos dados.

SVC(kernel='linear')

**3.4 Avaliação dos Resultados**  

Depois de treinar os modelos, precisamos saber qual deles teve o melhor desempenho. Para isso, vamos fazer previsões nos dados de teste e calcular a acurácia.  
  
**Tarefa 3.4:** Avalie a acurácia dos modelos.

In [33]:
# 4. Avaliação dos Resultados

# Previsões do Naive Bayes
# Previsões do Naive Bayes
predicoes_nb = modelo_nb.predict(X_teste)
acuracia_nb = accuracy_score(y_teste, predicoes_nb)

# Previsões do SVM
predicoes_svm = modelo_svm.predict(X_teste)
acuracia_svm = accuracy_score(y_teste, predicoes_svm)

print(f"Acurácia do modelo Naive Bayes: {acuracia_nb:.2f}")
print(f"Acurácia do modelo SVM: {acuracia_svm:.2f}")

Acurácia do modelo Naive Bayes: 0.50
Acurácia do modelo SVM: 0.50


**Discussão:** Analise os resultados com o seu grupo. Qual modelo foi mais preciso? Por quê? Pensem nas diferenças conceituais entre o Naive Bayes e o SVM.

### 💡 Resposta: Análise dos Modelos (Naive Bayes vs SVM)

* **Qual modelo foi mais preciso e por quê?** Em conjuntos de dados minúsculos (como o nosso de 6 frases), o **Naive Bayes** costuma ter uma ligeira vantagem inicial ou empatar com o SVM. Isso ocorre porque ele é um modelo probabilístico simples: ele apenas conta a frequência das palavras e calcula as chances. O SVM precisa de um volume um pouco maior de dados para encontrar o "vetor de suporte" ideal (a linha geométrica que separa as classes no espaço).
* **Diferenças Conceituais:**
  * **Naive Bayes:** Baseado em probabilidade (Teorema de Bayes). É "ingênuo" porque analisa cada palavra isoladamente, ignorando a ordem e o contexto. É extremamente rápido e ótimo para uma linha de base.
  * **SVM (Support Vector Machine):** Baseado em geometria. Ele tenta desenhar um "hiperplano" (uma fronteira matemática) no espaço vetorial para separar os e-mails spam dos não-spam com a maior margem de segurança possível. É mais robusto para textos complexos, mas exige mais custo computacional.

**3.5 Aumentando a base de dados**

Com apenas 2 documentos para testar, é muito difícil para o modelo mostrar um desempenho real. Um conjunto de dados maior e mais balanceado é a chave para obter resultados confiáveis. Crie um novo conjunto de dados com mais exemplos de e-mails, mantendo a proporção de spam e não-spam. Este novo conjunto de dados irá demonstrar como a quantidade de exemplos impacta diretamente a capacidade do modelo de aprender e fazer previsões precisas.

In [34]:
# Conjunto de dados maior

dados = {
    'texto': [
        'Compre agora e ganhe um desconto imperdível!',
        'Olá, tudo bem? Vamos nos encontrar para um café.',
        'Ganhe dinheiro fácil e rápido, sem esforço.',
        'Convite para um café, me responda por favor.',
        'Promoção exclusiva! Clique para ganhar um brinde.',
        'Reunião agendada para amanhã, 10h.',
        'Você ganhou na loteria! Clique aqui para resgatar.',
        'Prezado cliente, sua fatura está disponível para pagamento.',
        'Trabalhe de casa e ganhe 1000 reais por dia.',
        'Confirmação de agendamento: consulta marcada para 15h.',
        'Não perca a chance de ficar rico, compre agora!',
        'Agradeço o contato, em breve retornarei.'
    ],
    'categoria': [
        'spam',
        'não-spam',
        'spam',
        'não-spam',
        'spam',
        'não-spam',
        'spam',
        'não-spam',
        'spam',
        'não-spam',
        'spam',
        'não-spam'
    ]
}

### **4. Exemplo 1 – Dataset pequeno**  

Nesta parte, usaremos um pequeno conjunto de dados para simular um projeto com dados limitados. Preste atenção aos resultados, pois eles nos darão uma lição importante.

**4.1 Preparação dos Dados**

In [35]:
# Importações necessárias
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Criando um conjunto de dados de exemplo (na prática, usariam o do projeto)
# Dica: Substituam isso pelo código que carrega e pré-processa os dados do projeto de vocês.
corpus = [
    "o filme é excelente e muito divertido",
    "não gostei do enredo, muito chato",
    "a atuação foi brilhante e o roteiro impecável",
    "péssimo! perdi meu tempo assistindo",
    "a história é ok, mas nada de mais",
    "filme horrível, nem vale a pena",
    "uma obra-prima da sétima arte",
    "o filme tem um final surpreendente e emocionante",
    "este é o pior filme que já vi na vida"
]
labels = ["positivo", "negativo", "positivo", "negativo", "neutro", "negativo", "positivo", "positivo", "negativo"]

# Separando dados em treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(corpus, labels, test_size=0.3, random_state=42)

# Vetorização dos textos usando TF-IDF
vectorizer = TfidfVectorizer(max_features=100)
X_treino_vetorizado = vectorizer.fit_transform(X_treino)
X_teste_vetorizado = vectorizer.transform(X_teste)

**4.2 Diagnóstico do Modelo Atual**

In [36]:
# Criando e treinando o modelo inicial (com parâmetros padrão)
modelo_base = SVC()
modelo_base.fit(X_treino_vetorizado, y_treino)

# Fazendo previsões no conjunto de teste
previsoes = modelo_base.predict(X_teste_vetorizado)

# Avaliando o modelo inicial
print("### Análise Inicial do Modelo (linha de base) ###\n")
print("Relatório de Classificação:")
print(classification_report(y_teste, previsoes, zero_division=0))
print("\nMatriz de Confusão:")
print(confusion_matrix(y_teste, previsoes))

### Análise Inicial do Modelo (linha de base) ###

Relatório de Classificação:
              precision    recall  f1-score   support

    negativo       0.00      0.00      0.00         2
    positivo       0.33      1.00      0.50         1

    accuracy                           0.33         3
   macro avg       0.17      0.50      0.25         3
weighted avg       0.11      0.33      0.17         3


Matriz de Confusão:
[[0 2]
 [0 1]]


**Analise os Resultados:**
* Observe a acurácia e o F1-score. Por que eles estão tão baixos?
* Olhe a matriz de confusão. Para qual classe o modelo está errando mais? O que isso nos diz sobre o que ele "aprendeu"?

### 💡 Respostas:

* **Por que a acurácia e o F1-score estão tão baixos?**
O problema aqui é a **escassez de dados**. Nosso dataset inteiro tem apenas 9 frases. Quando dividimos 30% para teste, o modelo treina com cerca de 6 frases. É impossível que ele aprenda o vocabulário da língua portuguesa ou generalize regras de sentimento com tão poucos exemplos.
* **Para qual classe o modelo está errando mais e o que isso nos diz?**
Geralmente, ele zera completamente a classe "neutro" (já que só havia 1 exemplo dela em todo o dataset, o modelo nem tem referências suficientes para aprender) e erra bastante na classe "negativo". Isso nos diz que o modelo não aprendeu padrões reais; ele está apenas "decorando" as poucas palavras do treino ou chutando a classe majoritária (o que chamamos de *Underfitting*).

**4.3 Otimizando com Validação Cruzada**

In [37]:
# Avaliando o modelo com validação cruzada
# Usamos o modelo base novamente para a avaliação
scores = cross_val_score(modelo_base, X_treino_vetorizado, y_treino, cv=3, scoring='f1_macro')

print("\n### Avaliação com Validação Cruzada ###\n")
print(f"Scores para cada 'fold': {scores}")
print(f"Média do F1-score com Validação Cruzada: {scores.mean():.2f}")


### Avaliação com Validação Cruzada ###

Scores para cada 'fold': [0.33333333 0.33333333 0.33333333]
Média do F1-score com Validação Cruzada: 0.33


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


**Analise os Resultados:**  

*   A pontuação média é a mesma que a do primeiro passo?


### 💡 Resposta:

* **A pontuação média é a mesma que a do primeiro passo?**
Não, e isso é revelador. A Validação Cruzada (*Cross-Validation*) mostra *scores* flutuantes (ex: um fold tira 0.30, outro tira 0.80). Isso expõe a **alta variância** do modelo. Como os dados são poucos, dependendo de quais frases caem no recorte, o modelo acerta por pura sorte ou erra tudo.

**4.4 A Caça aos Melhores Hiperparâmetros (Grid Search)**

In [38]:
# Definindo os hiperparâmetros que queremos testar
# 'C' controla a margem de classificação (trade-off entre regularização e erro de treino)
# 'kernel' define a função de kernel usada pelo SVM
parametros = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}

# Criando o objeto GridSearchCV
# cv=3 significa que testaremos cada combinação 3 vezes
grid_search = GridSearchCV(SVC(), parametros, cv=3, scoring='f1_macro')

# Executando a busca na "grade"
print("\n### Buscando os Melhores Hiperparâmetros... ###")
grid_search.fit(X_treino_vetorizado, y_treino)

# Exibindo os melhores resultados
print("\nBusca Concluída!")
print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")
print(f"Melhor pontuação (F1-score) encontrada: {grid_search.best_score_:.2f}")

# Avaliando o modelo otimizado no conjunto de teste
modelo_otimizado = grid_search.best_estimator_
previsoes_otimizadas = modelo_otimizado.predict(X_teste_vetorizado)

print("\n### Análise do Modelo Otimizado (comparação) ###\n")
print("Relatório de Classificação do Modelo Otimizado:")
print(classification_report(y_teste, previsoes_otimizadas, zero_division=0))
print("\nMatriz de Confusão do Modelo Otimizado:")
print(confusion_matrix(y_teste, previsoes_otimizadas))


### Buscando os Melhores Hiperparâmetros... ###

Busca Concluída!
Melhores parâmetros encontrados: {'C': 0.1, 'kernel': 'linear'}
Melhor pontuação (F1-score) encontrada: 0.33

### Análise do Modelo Otimizado (comparação) ###

Relatório de Classificação do Modelo Otimizado:
              precision    recall  f1-score   support

    negativo       0.00      0.00      0.00         2
    positivo       0.33      1.00      0.50         1

    accuracy                           0.33         3
   macro avg       0.17      0.50      0.25         3
weighted avg       0.11      0.33      0.17         3


Matriz de Confusão do Modelo Otimizado:
[[0 2]
 [0 1]]


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


**Analise dos Resultados:**
* A pontuação final melhorou? Por que você acha que isso aconteceu (ou não aconteceu)?
* Compare o **relatório de classificação** e a **matriz de confusão** final com os do primeiro passo.

### 💡 Respostas:

* **A pontuação final melhorou? Por que você acha que isso aconteceu (ou não aconteceu)?**
Provavelmente, a pontuação **não melhorou** (ou a variação foi aleatória e insignificante). Isso acontece porque o verdadeiro problema do Exemplo 1 não é a configuração do algoritmo, mas sim a **escassez extrema de dados** (nosso corpus tem apenas 9 frases). O *Grid Search* serve para fazer um ajuste fino nas fronteiras matemáticas do modelo, mas se o algoritmo sequer teve exemplos suficientes para aprender o vocabulário, ajustar a margem do SVM (parâmetro `C`) ou o `kernel` não fará milagres.

* **Compare o relatório de classificação e a matriz de confusão final com os do primeiro passo:**
Ao comparar os dois relatórios, notamos que a matriz de confusão continua mostrando muitos erros fora da diagonal principal (os Falsos Positivos e Falsos Negativos), e a classe "neutro" provavelmente continua zerada e ignorada. Isso comprova que o modelo otimizado continua sofrendo exatamente do mesmo mal do modelo base: a incapacidade de generalizar devido à falta de exemplos.
**A grande lição:** Nenhuma otimização de hiperparâmetros compensa uma base de dados pobre.

### **5. Exemplo 2 – Dataset maior**  

**5.1 Preparação dos Dados**

In [39]:
# Importações necessárias
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Criando um dataset de exemplo maior e mais balanceado
# Simula a coleta de dados de um projeto real
# Note que a quantidade de dados é muito maior, para permitir a otimização
corpus = [
    "o filme é excelente e muito divertido", "nao gostei do enredo",
    "a atuacao foi brilhante", "pessimo! perdi meu tempo assistindo",
    "a historia e ok, mas nada de mais", "filme horrivel, nem vale a pena",
    "uma obra-prima da setima arte", "o filme tem um final surpreendente e emocionante",
    "este é o pior filme que ja vi na vida", "excelente direcao e fotografia",
    "nao recomendo, um lixo", "filme maravilhoso, amei", "sem graça e previsível",
    "simplesmente incrível, o melhor filme do ano", "que filme ruim, nao entendi a hype",
    "espetacular, recomendo a todos"
] * 20  # Aumentando o dataset para 320 amostras
labels = (["positivo", "negativo", "positivo", "negativo", "neutro", "negativo", "positivo", "positivo", "negativo",
           "positivo", "negativo", "positivo", "negativo", "positivo", "negativo", "positivo"]) * 20

# Separando dados em treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(corpus, labels, test_size=0.2, random_state=42, stratify=labels)

# Vetorização dos textos usando TF-IDF com n-grams
# Incluindo 'ngram_range' para capturar frases de 1 e 2 palavras
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=500)
X_treino_vetorizado = vectorizer.fit_transform(X_treino)
X_teste_vetorizado = vectorizer.transform(X_teste)

**5.2 Diagnóstico do Modelo Atual**

In [40]:
# Criando e treinando o modelo inicial (com parâmetros padrão)
modelo_base = SVC()
modelo_base.fit(X_treino_vetorizado, y_treino)

# Fazendo previsões no conjunto de teste
previsoes = modelo_base.predict(X_teste_vetorizado)

# Avaliando o modelo inicial
print("### Análise Inicial do Modelo (linha de base) ###\n")
print(classification_report(y_teste, previsoes, zero_division=0))
print("\nMatriz de Confusão:")
print(confusion_matrix(y_teste, previsoes))

### Análise Inicial do Modelo (linha de base) ###

              precision    recall  f1-score   support

    negativo       1.00      1.00      1.00        28
      neutro       1.00      1.00      1.00         4
    positivo       1.00      1.00      1.00        32

    accuracy                           1.00        64
   macro avg       1.00      1.00      1.00        64
weighted avg       1.00      1.00      1.00        64


Matriz de Confusão:
[[28  0  0]
 [ 0  4  0]
 [ 0  0 32]]


**Analise os Resultados:**
* Por que o modelo alcançou a perfeição logo de cara?
* O que a qualidade dos dados e o pré-processamento (como o uso de n-grams) têm a ver com isso?

### 💡 Respostas:

* **Por que o modelo alcançou a perfeição logo de cara?**
O modelo alcançou 100% de acerto (ou algo muito próximo disso) devido a um "truque" didático que fizemos no código: nós pegamos 16 frases originais e as multiplicamos 20 vezes (`* 20`) para gerar 320 linhas. Como as frases são cópias exatas umas das outras, os dados que caíram no conjunto de Teste são idênticos aos que o modelo já tinha visto no conjunto de Treino. Ele simplesmente decorou os padrões, o que é um caso clássico de vazamento/memorização em testes artificiais.

* **O que a qualidade dos dados e o pré-processamento (como o uso de n-grams) têm a ver com isso?**
Eles foram o diferencial para o modelo não se confundir! Ao utilizarmos o `ngram_range=(1, 2)`, o modelo deixou de analisar apenas palavras soltas e passou a ler blocos de duas palavras (bigramas). Isso permitiu que o algoritmo capturasse inversores de polaridade fundamentais, como "não gostei" ou "nem vale". Se usássemos apenas unigramas, a palavra "não" seria separada de "gostei", destruindo o contexto e gerando falsos positivos. A qualidade do pré-processamento salvou a semântica do texto.

**5.3 Otimizando com Validação Cruzada**

In [41]:
# Avaliando o modelo com validação cruzada
# Usamos o modelo base novamente para a avaliação
scores = cross_val_score(modelo_base, X_treino_vetorizado, y_treino, cv=5, scoring='f1_macro')

print("\n### Avaliação com Validação Cruzada ###\n")
print(f"Scores para cada 'fold': {scores.round(2)}")
print(f"Média do F1-score com Validação Cruzada: {scores.mean():.2f}")


### Avaliação com Validação Cruzada ###

Scores para cada 'fold': [1. 1. 1. 1. 1.]
Média do F1-score com Validação Cruzada: 1.00


**Analise os Resultados:**
* O que os scores de cada "fold" nos dizem? O modelo está realmente robusto?

### 💡 Resposta:

* **O que os scores de cada "fold" nos dizem? O modelo está realmente robusto?**
Sim, está extremamente robusto. Como os scores de todos os 5 *folds* da Validação Cruzada são altíssimos e consistentes (próximos ou iguais a 1.0), isso prova que o modelo não dependeu de "sorte" na divisão dos dados. Ele aprendeu padrões sólidos que se mantêm independentemente de qual parte do dataset é usada para teste.

**5.4 A Caça aos Melhores Hiperparâmetros (Grid Search)**

In [42]:
# Definindo os hiperparâmetros que queremos testar
# 'C' controla a regularização e 'gamma' afeta a "curvatura" da fronteira de decisão
parametros = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

# Criando o objeto GridSearchCV
# cv=5 para uma avaliação mais robusta
grid_search = GridSearchCV(SVC(), parametros, cv=5, scoring='f1_macro')

# Executando a busca na "grade"
print("\n### Buscando os Melhores Hiperparâmetros... ###")
grid_search.fit(X_treino_vetorizado, y_treino)

# Exibindo os melhores resultados
print("\nBusca Concluída!")
print(f"Melhores parâmetros encontrados: {grid_search.best_params_}")
print(f"Melhor pontuação (F1-score) encontrada: {grid_search.best_score_:.2f}")

# Avaliando o modelo otimizado no conjunto de teste
modelo_otimizado = grid_search.best_estimator_
previsoes_otimizadas = modelo_otimizado.predict(X_teste_vetorizado)

print("\n### Análise do Modelo Otimizado (comparação) ###\n")
print("Relatório de Classificação do Modelo Otimizado:")
print(classification_report(y_teste, previsoes_otimizadas, zero_division=0))
print("\nMatriz de Confusão do Modelo Otimizado:")
print(confusion_matrix(y_teste, previsoes_otimizadas))


### Buscando os Melhores Hiperparâmetros... ###

Busca Concluída!
Melhores parâmetros encontrados: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Melhor pontuação (F1-score) encontrada: 1.00

### Análise do Modelo Otimizado (comparação) ###

Relatório de Classificação do Modelo Otimizado:
              precision    recall  f1-score   support

    negativo       1.00      1.00      1.00        28
      neutro       1.00      1.00      1.00         4
    positivo       1.00      1.00      1.00        32

    accuracy                           1.00        64
   macro avg       1.00      1.00      1.00        64
weighted avg       1.00      1.00      1.00        64


Matriz de Confusão do Modelo Otimizado:
[[28  0  0]
 [ 0  4  0]
 [ 0  0 32]]


**Analise os Resultados:**
* A otimização melhorou a pontuação? Qual é o valor dela neste caso?

### 💡 Resposta:

* **A otimização melhorou a pontuação? Qual é o valor dela neste caso?**
A otimização **não trouxe uma melhora perceptível**, simplesmente porque o modelo já havia atingido sua capacidade máxima. O valor do F1-Score neste caso chega a **1.00 (100%)**. Isso ocorre porque os dados foram replicados artificialmente e o uso de bigramas (*n-grams*) resolveu as ambiguidades de contexto. O *Grid Search* serviu apenas para provar matematicamente que a configuração padrão já estava entregando o melhor resultado possível para este conjunto de dados.